In [1]:
import pandas as pd 
import matplotlib.pyplot as plt

In [2]:
df=pd.read_csv('PV_Diagnostic_Cleaned.csv',sep=';')
df

,timestamp,I1,I2,V1,V2,Irr,T1,T2,P1,P2,Label_Num_S1,Label_Num_S2,Label_S1,Label_S2
0,2026-01-10 09:29:16.310880,7.750000,0.243125,0.036562,0.034688,978.0,45.000000,44.258552,0.283359,0.008433,2,1,Short Circuit,Normal
1,2026-01-10 09:29:17.360277,1.167500,0.248750,0.031875,0.040313,935.0,45.000000,44.258552,0.037214,0.010028,1,1,Normal,Normal
2,2026-01-10 09:29:18.419751,1.173125,0.260000,0.025313,0.015938,935.0,45.000000,44.258552,0.029695,0.004144,1,1,Normal,Normal
3,2026-01-10 09:29:19.472936,1.169375,0.252500,0.036562,0.036562,900.0,45.000000,44.258552,0.042755,0.009232,1,1,Normal,Normal
4,2026-01-10 09:29:20.529135,1.180625,0.258125,0.034688,0.033750,900.0,45.000000,44.292574,0.040953,0.008712,1,1,Normal,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226170,2026-01-23 12:25:11.229436,0.773750,0.541250,15.423750,15.245625,131.0,24.956324,26.309621,11.934127,8.251695,1,4,Normal,Degradation
226171,2026-01-23 12:25:12.291174,0.764375,0.537500,15.424687,15.241875,130.0,24.922499,26.309621,11.790246,8.192508,1,4,Normal,Degradation
226172,2026-01-23 12:25:13.357950,0.777500,0.526250,15.426562,15.241875,130.0,24.922499,26.309621,11.994152,8.021037,1,4,Normal,Degradation
226173,2026-01-23 12:25:14.413214,0.777500,0.535625,15.427500,15.243750,130.0,24.922499,26.275782,11.994881,8.164934,1,4,Normal,Degradation


In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

In [4]:
# =========================
# FEATURES
# =========================

features_s1 = [
    "I1",
    "V1",
    "T1",
    "P1",
    "Irr"
]

features_s2 = [
    "I2",
    "V2",
    "T2",
    "P2",
    "Irr"
]

print("S1 features:", features_s1)
print("S2 features:", features_s2)

S1 features: ['I1', 'V1', 'T1', 'P1', 'Irr']
S2 features: ['I2', 'V2', 'T2', 'P2', 'Irr']


In [5]:
# =========================
# S1 DATA
# =========================

data_s1 = df[
    features_s1 + ["Label_Num_S1"]
].dropna().copy()

X_s1 = data_s1[features_s1]

# Original labels: 1,2,3,4,5
y_s1 = data_s1["Label_Num_S1"]

print("S1 original labels:")
print(sorted(y_s1.unique()))

S1 original labels:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [6]:
# Convert 1,2,3,4,5 -> 0,1,2,3,4
y_s1_encoded = y_s1 - 1

print("S1 labels used for XGBoost:")
print(sorted(y_s1_encoded.unique()))

S1 labels used for XGBoost:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [7]:
X_train_s1, X_test_s1, y_train_s1, y_test_s1 = train_test_split(
    X_s1,
    y_s1_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_s1_encoded
)

In [8]:
# =========================
# XGBOOST S1
# =========================

model_s1 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(y_s1_encoded.unique()),
    eval_metric="mlogloss",
    random_state=42
)

model_s1.fit(
    X_train_s1,
    y_train_s1
)

print("S1 model trained successfully!")

S1 model trained successfully!


In [9]:
y_pred_s1 = model_s1.predict(X_test_s1)

print(
    "S1 Accuracy:",
    accuracy_score(y_test_s1, y_pred_s1)
)

print("\nS1 Classification Report:")

print(
    classification_report(
        y_test_s1,
        y_pred_s1
    )
)

S1 Accuracy: 0.9965513429866254

S1 Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9461
           1       0.88      0.64      0.74        11
           2       1.00      1.00      1.00     11634
           3       1.00      1.00      1.00     20777
           4       1.00      1.00      1.00      3352

    accuracy                           1.00     45235
   macro avg       0.97      0.92      0.94     45235
weighted avg       1.00      1.00      1.00     45235



In [10]:
y_pred_s1_original = y_pred_s1 + 1

print("Original S1 predictions:")
print(y_pred_s1_original[:20])

Original S1 predictions:
[1 4 3 4 1 1 4 3 1 3 1 3 4 4 5 1 3 3 4 1]


In [11]:
# =========================
# S2 DATA
# =========================

data_s2 = df[
    features_s2 + ["Label_Num_S2"]
].dropna().copy()

X_s2 = data_s2[features_s2]

# Original labels
y_s2 = data_s2["Label_Num_S2"]

print("S2 original labels:")
print(sorted(y_s2.unique()))

S2 original labels:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [12]:
# Convert 1,2,3,4,5 -> 0,1,2,3,4
y_s2_encoded = y_s2 - 1

print("S2 labels used for XGBoost:")
print(sorted(y_s2_encoded.unique()))

S2 labels used for XGBoost:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [13]:
X_train_s2, X_test_s2, y_train_s2, y_test_s2 = train_test_split(
    X_s2,
    y_s2_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_s2_encoded
)

In [14]:
# =========================
# XGBOOST S2
# =========================

model_s2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(y_s2_encoded.unique()),
    eval_metric="mlogloss",
    random_state=42
)

model_s2.fit(
    X_train_s2,
    y_train_s2
)

print("S2 model trained successfully!")

S2 model trained successfully!


In [15]:
y_pred_s2 = model_s2.predict(X_test_s2)

print(
    "S2 Accuracy:",
    accuracy_score(y_test_s2, y_pred_s2)
)

print("\nS2 Classification Report:")

print(
    classification_report(
        y_test_s2,
        y_pred_s2
    )
)

S2 Accuracy: 0.9902509119044988

S2 Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     10373
           1       0.88      0.85      0.87        27
           2       1.00      1.00      1.00     10421
           3       0.99      0.99      0.99     21134
           4       0.99      1.00      1.00      3280

    accuracy                           0.99     45235
   macro avg       0.97      0.96      0.97     45235
weighted avg       0.99      0.99      0.99     45235



In [16]:
# ==========================================================
# CHECK MODEL OVERFITTING (TRAIN vs TEST ACCURACY GAP)
# ==========================================================

# --- Panel 1 (S1) ---
train_acc_s1 = accuracy_score(y_train_s1, model_s1.predict(X_train_s1))
test_acc_s1 = accuracy_score(y_test_s1, y_pred_s1)
gap_s1 = abs(train_acc_s1 - test_acc_s1) * 100

print("=== Panel 1 (S1) Overfitting Check ===")
print(f"Train Accuracy:     {train_acc_s1 * 100:.2f}%")
print(f"Test Accuracy:      {test_acc_s1 * 100:.2f}%")
print(f"Generalization Gap: {gap_s1:.2f}%")
if gap_s1 < 1.0:
    print("Verdict: ✅ No Overfitting (Model generalizes well)\n")
elif gap_s1 < 3.0:
    print("Verdict: ⚠️ Acceptable (Mild gap)\n")
else:
    print("Verdict: ❌ Overfitting Detected\n")

# --- Panel 2 (S2) ---
train_acc_s2 = accuracy_score(y_train_s2, model_s2.predict(X_train_s2))
test_acc_s2 = accuracy_score(y_test_s2, y_pred_s2)
gap_s2 = abs(train_acc_s2 - test_acc_s2) * 100

print("=== Panel 2 (S2) Overfitting Check ===")
print(f"Train Accuracy:     {train_acc_s2 * 100:.2f}%")
print(f"Test Accuracy:      {test_acc_s2 * 100:.2f}%")
print(f"Generalization Gap: {gap_s2:.2f}%")
if gap_s2 < 1.0:
    print("Verdict: ✅ No Overfitting (Model generalizes well)")
elif gap_s2 < 3.0:
    print("Verdict: ⚠️ Acceptable (Mild gap)")
else:
    print("Verdict: ❌ Overfitting Detected")


=== Panel 1 (S1) Overfitting Check ===
Train Accuracy:     99.80%
Test Accuracy:      99.66%
Generalization Gap: 0.14%
Verdict: ✅ No Overfitting (Model generalizes well)

=== Panel 2 (S2) Overfitting Check ===
Train Accuracy:     99.20%
Test Accuracy:      99.03%
Generalization Gap: 0.17%
Verdict: ✅ No Overfitting (Model generalizes well)


In [17]:
print(
    df[
        ["Label_Num_S1", "Label_S1"]
    ]
    .drop_duplicates()
    .sort_values("Label_Num_S1")
    .to_string(index=False)
)

 Label_Num_S1      Label_S1
            1        Normal
            2 Short Circuit
            3  Open Circuit
            4   Degradation
            5       Shading


In [18]:
print(
    df[
        ["Label_Num_S2", "Label_S2"]
    ]
    .drop_duplicates()
    .sort_values("Label_Num_S2")
    .to_string(index=False)
)

 Label_Num_S2      Label_S2
            1        Normal
            2 Short Circuit
            3  Open Circuit
            4   Degradation
            5       Shading


In [19]:
# ==========================================================
# CHECK MODEL OVERFITTING (TRAIN vs TEST ACCURACY)
# ==========================================================

# --- Panel 1 (S1) ---
train_acc_s1 = accuracy_score(y_train_s1, model_s1.predict(X_train_s1))
test_acc_s1 = accuracy_score(y_test_s1, y_pred_s1)
gap_s1 = abs(train_acc_s1 - test_acc_s1) * 100

print("=== Panel 1 (S1) Overfitting Check ===")
print(f"Train Accuracy:     {train_acc_s1 * 100:.2f}%")
print(f"Test Accuracy:      {test_acc_s1 * 100:.2f}%")
print(f"Generalization Gap: {gap_s1:.2f}%")
if gap_s1 < 1.0:
    print("Verdict: ✅ No Overfitting (Model generalizes well)\n")
elif gap_s1 < 3.0:
    print("Verdict: ⚠️ Acceptable (Mild gap)\n")
else:
    print("Verdict: ❌ Overfitting Detected\n")

# --- Panel 2 (S2) ---
train_acc_s2 = accuracy_score(y_train_s2, model_s2.predict(X_train_s2))
test_acc_s2 = accuracy_score(y_test_s2, y_pred_s2)
gap_s2 = abs(train_acc_s2 - test_acc_s2) * 100

print("=== Panel 2 (S2) Overfitting Check ===")
print(f"Train Accuracy:     {train_acc_s2 * 100:.2f}%")
print(f"Test Accuracy:      {test_acc_s2 * 100:.2f}%")
print(f"Generalization Gap: {gap_s2:.2f}%")
if gap_s2 < 1.0:
    print("Verdict: ✅ No Overfitting (Model generalizes well)")
elif gap_s2 < 3.0:
    print("Verdict: ⚠️ Acceptable (Mild gap)")
else:
    print("Verdict: ❌ Overfitting Detected")


=== Panel 1 (S1) Overfitting Check ===
Train Accuracy:     99.80%
Test Accuracy:      99.66%
Generalization Gap: 0.14%
Verdict: ✅ No Overfitting (Model generalizes well)

=== Panel 2 (S2) Overfitting Check ===
Train Accuracy:     99.20%
Test Accuracy:      99.03%
Generalization Gap: 0.17%
Verdict: ✅ No Overfitting (Model generalizes well)


In [20]:
# =========================
# SAVE S1
# =========================

package_s1 = {
    "model": model_s1,
    "features": features_s1,
    "label_offset": 1
}

joblib.dump(
    package_s1,
    "pv_xgboost_s1.pkl"
)


# =========================
# SAVE S2
# =========================

package_s2 = {
    "model": model_s2,
    "features": features_s2,
    "label_offset": 1
}

joblib.dump(
    package_s2,
    "pv_xgboost_s2.pkl"
)

print("Both models saved successfully!")

Both models saved successfully!


In [21]:
package_s1 = joblib.load("pv_xgboost_s1.pkl")
package_s2 = joblib.load("pv_xgboost_s2.pkl")

model_s1 = package_s1["model"]
features_s1 = package_s1["features"]

model_s2 = package_s2["model"]
features_s2 = package_s2["features"]

print("Models loaded successfully!")

Models loaded successfully!


In [22]:
new_data = pd.DataFrame([{
    "I1": 5.2,
    "I2": 5.1,
    "V1": 38.5,
    "V2": 38.2,
    "Irr": 850,
    "T1": 42.3,
    "T2": 43.1,
    "P1": 200.1,
    "P2": 195.4
}])

In [23]:
pred_s1 = model_s1.predict(
    new_data[features_s1]
)

pred_s1_original = pred_s1 + 1

print("S1 predicted label:", pred_s1_original[0])

S1 predicted label: 1


In [24]:
pred_s2 = model_s2.predict(
    new_data[features_s2]
)

pred_s2_original = pred_s2 + 1

print("S2 predicted label:", pred_s2_original[0])

S2 predicted label: 1


In [25]:
new_data = pd.DataFrame([{
    "I1": 5.2,
    "I2": 5.1,
    "V1": 38.5,
    "V2": 38.2,
    "Irr": 850,
    "T1": 42.3,
    "T2": 43.1,
    "P1": 200.1,
    "P2": 195.4
}])

# =========================
# S1 Prediction
# =========================

pred_s1 = model_s1.predict(
    new_data[features_s1]
)

pred_s1_original = pred_s1 + 1

print("S1 predicted label:", pred_s1_original[0])


# =========================
# S2 Prediction
# =========================

pred_s2 = model_s2.predict(
    new_data[features_s2]
)

pred_s2_original = pred_s2 + 1

print("S2 predicted label:", pred_s2_original[0])

S1 predicted label: 1
S2 predicted label: 1


In [26]:
new_data = pd.DataFrame([{
    # ===== S1: Normal =====
    "I1": 5.2,
    "V1": 38.5,
    "T1": 42.3,
    "P1": 200.1,

    # ===== S2: Shading =====
    "I2": 2.0,
    "V2": 36.5,
    "T2": 43.1,
    "P2": 73.0,

    # Irradiance
    "Irr": 850
}])

# S1
pred_s1 = model_s1.predict(new_data[features_s1])
pred_s1_original = pred_s1 + 1

# S2
pred_s2 = model_s2.predict(new_data[features_s2])
pred_s2_original = pred_s2 + 1

label_names = {
    1: "Normal",
    2: "Short Circuit",
    3: "Open Circuit",
    4: "Degradation",
    5: "Shading"
}

print("S1 predicted:", label_names[int(pred_s1_original[0])])
print("S2 predicted:", label_names[int(pred_s2_original[0])])

S1 predicted: Normal
S2 predicted: Degradation
